In [1]:
!pip install xgboost lightgbm joblib scikit-learn pandas numpy matplotlib

In [2]:
import pandas as pd

df=pd.read_csv("scaled_fraud_training_data_100k.csv")

df.columns

Index(['transaction_id', 'customer_id', 'beneficiary_id', 'merchant_id',
       'device_id', 'transaction_timestamp', 'transaction_type',
       'transaction_amount', 'payment_method', 'ip_address', 'origin_country',
       'destination_country', 'transaction_status', 'is_international',
       'hour_of_day', 'account_age_days', 'transaction_frequency_24h',
       'failed_transaction_count_24h', 'avg_transaction_amount_7d',
       'session_duration_minutes', 'device_risk_score', 'unusual_amount_flag',
       'unusual_location_flag', 'typing_speed_flag',
       'shared_device_mule_count', 'known_fraud_ring_edge',
       'biometric_anomaly_detected', 'automation_script_suspected',
       'attack_vector_type', 'is_fraud', 'risk_score', 'risk_category'],
      dtype='object')

In [2]:
# !pip install xgboost lightgbm scikit-learn pandas numpy matplotlib

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error, r2_score

import xgboost as xgb
import lightgbm as lgb
import joblib

print("✅ Core ML transformer engines loaded.")

✅ Core ML transformer engines loaded.


In [3]:
DATA_PATH = "scaled_fraud_training_data_100k.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Missing dataset source! Please run the 100k generation script first.")

df = pd.read_csv(DATA_PATH)

# Define feature spaces based on required mathematical treatments
log_features = ["transaction_amount", "avg_transaction_amount_7d"]
cyclical_features = ["hour_of_day"]
numeric_features = ["account_age_days", "transaction_frequency_24h", "failed_transaction_count_24h", "session_duration_minutes", "device_risk_score", "shared_device_mule_count"]
passthrough_features = ["is_international", "unusual_amount_flag", "unusual_location_flag", "typing_speed_flag", "known_fraud_ring_edge", "biometric_anomaly_detected", "automation_script_suspected"]

all_features = log_features + cyclical_features + numeric_features + passthrough_features
X = df[all_features]
y_clf = df["is_fraud"]
y_reg = df["risk_score"]

# Split into train/test splits before transformations to maintain pure validation environments
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, stratify=y_clf, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f"📈 Splits initialized. Base feature matrix columns shape: {X.shape[1]}")

📈 Splits initialized. Base feature matrix columns shape: 16


In [4]:
def log_transform(x):
    return np.log1p(x)

def cyclical_hour_sine(x):
    return np.sin(2 * np.pi * x / 24.0)

def cyclical_hour_cosine(x):
    return np.cos(2 * np.pi * x / 24.0)

# Build custom Scikit-Learn compatible transformers
log_transformer = FunctionTransformer(log_transform, validate=True)
sin_transformer = FunctionTransformer(cyclical_hour_sine, validate=True)
cos_transformer = FunctionTransformer(cyclical_hour_cosine, validate=True)

print("🛠️ Custom mathematical transformations engineered.")

🛠️ Custom mathematical transformations engineered.


In [5]:
# Build isolated branch paths for cyclical parsing to create two output features from one input column
cyclical_pipeline = ColumnTransformer(
    transformers=[
        ('hour_sin', sin_transformer, [0]),
        ('hour_cos', cos_transformer, [0])
    ]
)

# Combine everything into a master preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('log_transform', Pipeline([('log', log_transformer), ('scale', StandardScaler())]), log_features),
        ('cyclical_time', cyclical_pipeline, cyclical_features),
        ('standard_scale', StandardScaler(), numeric_features),
        ('pass', 'passthrough', passthrough_features)
    ]
)

# Fit the transformer array securely on training data only
X_train_c_transformed = preprocessor.fit_transform(X_train_c)
X_test_c_transformed = preprocessor.transform(X_test_c)

print(f"🔄 Transformation pipeline fitted. Engineered feature width: {X_train_c_transformed.shape[1]}")

🔄 Transformation pipeline fitted. Engineered feature width: 17


In [6]:
print("⏳ Optimizing Classification Hyperparameters over Preprocessed Data Grid...")

xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)

param_grid_clf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.8, 0.9],
    'scale_pos_weight': [1, 3.5]
}

clf_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_grid_clf, n_iter=8, scoring='roc_auc', cv=3, verbose=1, random_state=42, n_jobs=-1)
clf_search.fit(X_train_c_transformed, y_train_c)

best_clf = clf_search.best_estimator_
print(f"🏆 Best Trained Classification Configuration:\n{json.dumps(clf_search.best_params_, indent=2)}")

⏳ Optimizing Classification Hyperparameters over Preprocessed Data Grid...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:57:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🏆 Best Trained Classification Configuration:
{
  "subsample": 0.9,
  "scale_pos_weight": 1,
  "n_estimators": 300,
  "max_depth": 4,
  "learning_rate": 0.2
}


In [7]:
y_pred_c = best_clf.predict(X_test_c_transformed)
y_prob_c = best_clf.predict_proba(X_test_c_transformed)[:, 1]

print("📊 --- Post-Transformation Classification Performance ---")
print(classification_report(y_test_c, y_pred_c))
print(f"Optimized AUC-ROC Metric Score: {roc_auc_score(y_test_c, y_prob_c):.4f}")

📊 --- Post-Transformation Classification Performance ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17194
           1       1.00      1.00      1.00      2806

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

Optimized AUC-ROC Metric Score: 1.0000


In [8]:
X_train_r_transformed = preprocessor.fit_transform(X_train_r)
X_test_r_transformed = preprocessor.transform(X_test_r)

lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)

param_grid_reg = {
    'n_estimators': [200, 400],
    'max_depth': [6, 8, 10],
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.01, 0.05, 0.1]
}

reg_search = RandomizedSearchCV(estimator=lgb_model, param_distributions=param_grid_reg, n_iter=8, scoring='neg_mean_absolute_error', cv=3, verbose=1, random_state=42, n_jobs=-1)
reg_search.fit(X_train_r_transformed, y_train_r)

best_reg = reg_search.best_estimator_
print(f"🏆 Best Trained Regressor Configuration:\n{json.dumps(reg_search.best_params_, indent=2)}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits
🏆 Best Trained Regressor Configuration:
{
  "num_leaves": 63,
  "n_estimators": 200,
  "max_depth": 6,
  "learning_rate": 0.05
}


In [9]:
y_pred_r = best_reg.predict(X_test_r_transformed)

print("📉 --- Post-Transformation Regression Performance ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_r, y_pred_r):.4f} index steps")
print(f"Model R-squared Reliability: {r2_score(y_test_r, y_pred_r)*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


📉 --- Post-Transformation Regression Performance ---
Mean Absolute Error (MAE): 9.0793 index steps
Model R-squared Reliability: 83.95%


In [10]:
# Serialize the pipeline components alongside the models
joblib.dump(preprocessor, 'production_preprocessor_pipeline.pkl')
joblib.dump(best_clf, 'transformed_fraud_classifier.pkl')
joblib.dump(best_reg, 'transformed_risk_regressor.pkl')

print("💾 All transformation states and hyperparameter-tuned weights successfully serialized to disk!")

💾 All transformation states and hyperparameter-tuned weights successfully serialized to disk!


# **Prediction**

In [11]:
import joblib
import pandas as pd
import numpy as np

# Load inference dependencies
loaded_preprocessor = joblib.load('production_preprocessor_pipeline.pkl')
loaded_classifier = joblib.load('transformed_fraud_classifier.pkl')
loaded_regressor = joblib.load('transformed_risk_regressor.pkl')

# Incoming unmapped mock streaming message payload direct from Kafka consumer
kafka_event_message = {
    "transaction_id": "TXN_LIVE_STREAM_99B",
    "transaction_amount": 22450.00,
    "is_international": 1,
    "hour_of_day": 3, # 3 AM Off-Hours
    "features_for_classifier": {
        "account_age_days": 5,
        "transaction_frequency_24h": 58,
        "failed_transaction_count_24h": 9,
        "avg_transaction_amount_7d": 120.00,
        "session_duration_minutes": 2,
        "device_risk_score": 98.40,
        "unusual_amount_flag": 1,
        "unusual_location_flag": 1,
        "typing_speed_flag": 1
    },
    "agent_pipelines_telemetry": {
        "graph_agent_context": {
            "shared_device_mule_count": 19,
            "known_fraud_ring_edge": 1
        },
        "behavioral_agent_context": {
            "biometric_anomaly_detected": 1,
            "automation_script_suspected": 1
        }
    }
}

def stream_realtime_inference(event):
    # Unpack nested components precisely into a single row DataFrame
    f_class = event["features_for_classifier"]
    g_agent = event["agent_pipelines_telemetry"]["graph_agent_context"]
    b_agent = event["agent_pipelines_telemetry"]["behavioral_agent_context"]

    raw_record = {
        "transaction_amount": event["transaction_amount"],
        "avg_transaction_amount_7d": f_class["avg_transaction_amount_7d"],
        "hour_of_day": event["hour_of_day"],
        "account_age_days": f_class["account_age_days"],
        "transaction_frequency_24h": f_class["transaction_frequency_24h"],
        "failed_transaction_count_24h": f_class["failed_transaction_count_24h"],
        "session_duration_minutes": f_class["session_duration_minutes"],
        "device_risk_score": f_class["device_risk_score"],
        "shared_device_mule_count": g_agent["shared_device_mule_count"],
        "is_international": event["is_international"],
        "unusual_amount_flag": f_class["unusual_amount_flag"],
        "unusual_location_flag": f_class["unusual_location_flag"],
        "typing_speed_flag": f_class["typing_speed_flag"],
        "known_fraud_ring_edge": g_agent["known_fraud_ring_edge"],
        "biometric_anomaly_detected": b_agent["biometric_anomaly_detected"],
        "automation_script_suspected": b_agent["automation_script_suspected"]
    }

    df_raw = pd.DataFrame([raw_record])

    # Transform raw event features through our pre-fitted transformer pipeline
    transformed_vector = loaded_preprocessor.transform(df_raw)

    # Compute downstream score actions
    is_fraud_decision = loaded_classifier.predict(transformed_vector)[0]
    predicted_score = loaded_regressor.predict(transformed_vector)[0]

    return {
        "transaction_id": event["transaction_id"],
        "system_action": "TERMINATE_TRANSACTION" if is_fraud_decision == 1 else "ALLOW_TRANSACTION",
        "realtime_risk_score": round(float(predicted_score), 2)
    }

# Execute inference process loop
output_payload = stream_realtime_inference(kafka_event_message)
print("📥 Real-Time Secure Process Inference Output:")
print(json.dumps(output_payload, indent=4))

📥 Real-Time Secure Process Inference Output:
{
    "transaction_id": "TXN_LIVE_STREAM_99B",
    "system_action": "TERMINATE_TRANSACTION",
    "realtime_risk_score": 95.81
}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
